# NB04: Monte Carlo regular-season finish simulation

- **Purpose:** estimate model-based final regular-season points-rank distributions by draft slot.
- **Pipeline:** saved historical panel -> slot-level point effects -> 17-week latent-score simulation -> final rank probabilities.
- **Inputs:** `data/processed/analysis_panel.csv` with 3,641 balanced league-seasons and 43,692 team-seasons.
- **Outputs:** simulation summary tables, assumptions, evaluation gates, and four interactive Plotly HTML charts under `artifacts/`.
- **Run:** execute after NB03 from the project root or the `notebooks/` directory.
- **Locked definitions:** 12 teams, 17 regular-season weeks, rank by total regular-season points, no head-to-head record, no playoffs, 100,000 simulated league-seasons per scenario, and seed 20260812.
- **Model boundary:** weekly scores are normalized latent deviations because the retained panel has season totals but no weekly matchup scores. Weekly persistence is therefore an explicit sensitivity assumption, not a historically estimated parameter.

| Gate | What it checks | Pass condition |
| ---: | --- | --- |
| 1 | Historical panel structure | 3,641 complete 12-team league-seasons and every slot present once |
| 2 | Rank accounting | Each scenario assigns one team to every rank in every simulation |
| 3 | Null symmetry | Null first-place rates remain within 0.5 percentage points of 1/12 |
| 4 | Monte Carlo precision | Maximum first-place standard error is below 0.15 percentage points |
| 5 | Artifact completeness | Summary files and four Plotly HTML charts are written |

### What this cell does

- Loads the retained panel and enforces the balanced 12-team league structure.
- Confirms the available seasons and total league-season coverage before calibration.

In [1]:
# CELL [1 load-and-validate-panel]
from pathlib import Path
import json
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
PANEL_PATH = ROOT / "data/processed/analysis_panel.csv"
ARTIFACTS = ROOT / "artifacts"
ARTIFACTS.mkdir(exist_ok=True)

panel = pd.read_csv(PANEL_PATH)
required = {"league_id", "season", "draft_slot", "points_zscore", "top_6_points", "top_regular_season_scorer"}
assert required.issubset(panel.columns), f"Missing columns: {sorted(required - set(panel.columns))}"
league_sizes = panel.groupby(["league_id", "season"]).size()
slot_counts = panel.groupby(["league_id", "season"])["draft_slot"].nunique()
assert len(panel) == 43_692
assert len(league_sizes) == 3_641
assert league_sizes.eq(12).all() and slot_counts.eq(12).all()
assert panel.groupby("draft_slot").size().eq(3_641).all()

print({
    "league_seasons": int(len(league_sizes)),
    "team_seasons": int(len(panel)),
    "seasons": sorted(panel["season"].unique().tolist()),
    "per_slot_n": int(panel.groupby("draft_slot").size().min()),
})

{'league_seasons': 3641, 'team_seasons': 43692, 'seasons': [2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025], 'per_slot_n': 3641}


### Interpreting the output

- The panel contains 3,641 complete league-seasons and 43,692 team-seasons from 2018 through 2025, with 3,641 observations per slot.
- Every league block has 12 teams and all 12 draft slots, so the structural gate passed.
- The balanced panel permits within-league calibration without unequal slot exposure.
- It does not establish that public Sleeper leagues represent all fantasy leagues.

### What this cell does

- Estimates pooled and 2022-2025 draft-slot effects from within-league points z-scores.
- Calibrates total residual variation from all 43,692 team-seasons.
- Locks the simulation count, seed, 17-week horizon, heavy-tail choice, and persistence scenarios.

In [2]:
# CELL [2 calibrate-model]
N_TEAMS = 12
N_WEEKS = 17
N_SIMULATIONS = 100_000
SEED = 20260812
T_DF = 5
BASELINE_ICC = 0.15
LOW_ICC = 0.05
HIGH_ICC = 0.30

pooled_effect = panel.groupby("draft_slot")["points_zscore"].mean().reindex(range(1, 13)).to_numpy()
recent_effect = panel.loc[panel["season"] >= 2022].groupby("draft_slot")["points_zscore"].mean().reindex(range(1, 13)).to_numpy()
observed_effect = panel["points_zscore"].to_numpy() - pooled_effect[panel["draft_slot"].to_numpy() - 1]
residual_sd = float(observed_effect.std(ddof=1))
assert abs(pooled_effect.sum()) < 1e-12
assert residual_sd > 0

scenario_specs = {
    "Pooled effects, ICC 0.15": (pooled_effect, BASELINE_ICC),
    "No slot effect, ICC 0.15": (np.zeros(N_TEAMS), BASELINE_ICC),
    "2022-2025 effects, ICC 0.15": (recent_effect, BASELINE_ICC),
    "Pooled effects, ICC 0.05": (pooled_effect, LOW_ICC),
    "Pooled effects, ICC 0.30": (pooled_effect, HIGH_ICC),
}

print({
    "pooled_best_slot": int(np.argmax(pooled_effect) + 1),
    "pooled_best_effect": round(float(pooled_effect.max()), 4),
    "pooled_slot_1_effect": round(float(pooled_effect[0]), 4),
    "recent_best_slot": int(np.argmax(recent_effect) + 1),
    "residual_sd": round(residual_sd, 4),
    "weekly_scores_available": False,
})

{'pooled_best_slot': 4, 'pooled_best_effect': 0.056, 'pooled_slot_1_effect': -0.1025, 'recent_best_slot': 4, 'residual_sd': 0.9985, 'weekly_scores_available': False}


### Interpreting the output

- Slot 4 has the largest pooled points effect at +0.056 z-score units; slot 1 is -0.102. Slot 4 also leads in the 2022-2025 calibration.
- Residual season-level variation is 0.9985 z-score units after removing pooled slot means, so draft slot explains little of total scoring variation.
- These estimates set the model's expected score differences by slot.
- Weekly scores are absent, so weekly persistence is assumed and sensitivity-tested rather than estimated.

### What this cell does

- Simulates complete 12-team leagues for 17 normalized scoring weeks.
- Uses a persistent team component plus heavy-tailed weekly shocks with five degrees of freedom.
- Converts final point totals into ranks and returns full rank probabilities for every draft slot.

In [3]:
# CELL [3 run-simulations]
def simulate_rank_distribution(slot_effects, weekly_icc):
    rng = np.random.default_rng(SEED)
    weekly_variance = residual_sd**2 / (N_WEEKS * (1 + (N_WEEKS - 1) * weekly_icc))
    persistent_sd = np.sqrt(weekly_icc * weekly_variance)
    shock_sd = np.sqrt((1 - weekly_icc) * weekly_variance)
    t_unit_scale = np.sqrt((T_DF - 2) / T_DF)

    persistent = rng.standard_normal((N_SIMULATIONS, N_TEAMS)) * persistent_sd
    totals = slot_effects[None, :] + N_WEEKS * persistent
    for _ in range(N_WEEKS):
        totals += rng.standard_t(T_DF, size=(N_SIMULATIONS, N_TEAMS)) * t_unit_scale * shock_sd

    order = np.argsort(-totals, axis=1)
    ranks = np.empty_like(order)
    ranks[np.arange(N_SIMULATIONS)[:, None], order] = np.arange(1, N_TEAMS + 1)
    rank_probabilities = np.column_stack([(ranks == rank).mean(axis=0) for rank in range(1, N_TEAMS + 1)])
    return ranks, rank_probabilities

simulation_results = {}
for scenario, (effects, icc) in scenario_specs.items():
    ranks, rank_probabilities = simulate_rank_distribution(effects, icc)
    simulation_results[scenario] = {"ranks": ranks, "rank_probabilities": rank_probabilities, "icc": icc}

baseline_name = "Pooled effects, ICC 0.15"
baseline_ranks = simulation_results[baseline_name]["ranks"]
baseline_probs = simulation_results[baseline_name]["rank_probabilities"]
print({
    "scenarios": len(simulation_results),
    "league_seasons_per_scenario": N_SIMULATIONS,
    "baseline_best_expected_finish_slot": int(np.argmin(baseline_ranks.mean(axis=0)) + 1),
    "baseline_highest_first_place_slot": int(np.argmax(baseline_probs[:, 0]) + 1),
})

{'scenarios': 5, 'league_seasons_per_scenario': 100000, 'baseline_best_expected_finish_slot': 4, 'baseline_highest_first_place_slot': 3}


### Interpreting the output

- All five scenarios completed 100,000 simulated 12-team league-seasons.
- Slot 4 has the best baseline expected finish. Slots 3 and 4 have nearly identical first-place probabilities, with their small ordering difference inside Monte Carlo error.
- The simulations provide full final points-rank distributions under pooled, null, recent, and persistence alternatives.
- They do not simulate head-to-head schedules, wins, waivers, trades, or playoffs.

### What these tests guard

- Verifies complete rank accounting within every scenario.
- Uses the no-slot-effect scenario as a symmetry test for the simulation harness.
- Calculates Monte Carlo standard errors and builds the inference-ready summary tables.

In [4]:
# CELL [4 validate-and-summarize]
summary_rows = []
rank_rows = []
for scenario, result in simulation_results.items():
    ranks = result["ranks"]
    probs = result["rank_probabilities"]
    assert np.allclose(probs.sum(axis=1), 1.0)
    assert np.allclose(probs.sum(axis=0), 1.0)
    for slot in range(N_TEAMS):
        first_probability = float(probs[slot, 0])
        summary_rows.append({
            "scenario": scenario,
            "draft_slot": slot + 1,
            "weekly_icc": result["icc"],
            "expected_finish": float(ranks[:, slot].mean()),
            "median_finish": float(np.median(ranks[:, slot])),
            "top_6_probability": float((ranks[:, slot] <= 6).mean()),
            "first_place_probability": first_probability,
            "first_place_mc_se": float(np.sqrt(first_probability * (1 - first_probability) / N_SIMULATIONS)),
        })
        for rank in range(N_TEAMS):
            rank_rows.append({"scenario": scenario, "draft_slot": slot + 1, "final_rank": rank + 1, "probability": float(probs[slot, rank])})

summary = pd.DataFrame(summary_rows)
rank_distribution = pd.DataFrame(rank_rows)
null_first = summary.loc[summary["scenario"] == "No slot effect, ICC 0.15", "first_place_probability"]
max_null_deviation = float((null_first - 1 / N_TEAMS).abs().max())
max_mc_se = float(summary["first_place_mc_se"].max())
assert max_null_deviation < 0.005
assert max_mc_se < 0.0015

baseline_summary = summary[summary["scenario"] == baseline_name].copy()
best_row = baseline_summary.loc[baseline_summary["expected_finish"].idxmin()]
print({
    "rank_accounting_gate": "pass",
    "max_null_first_place_deviation_pp": round(100 * max_null_deviation, 3),
    "max_first_place_mc_se_pp": round(100 * max_mc_se, 3),
    "best_slot": int(best_row["draft_slot"]),
    "best_expected_finish": round(float(best_row["expected_finish"]), 3),
    "best_first_place_probability": round(float(best_row["first_place_probability"]), 4),
})

{'rank_accounting_gate': 'pass', 'max_null_first_place_deviation_pp': 0.254, 'max_first_place_mc_se_pp': 0.093, 'best_slot': 4, 'best_expected_finish': 6.316, 'best_first_place_probability': 0.0905}


### Reading the test result

- Rank accounting passed in all scenarios. The largest null first-place deviation is 0.254 percentage points, below the 0.5-point gate.
- The largest first-place Monte Carlo standard error is 0.093 percentage points, below the 0.15-point gate.
- Slot 4 has the best baseline expected finish at 6.316 and a modeled 9.05% first-place probability.
- Passing validates the simulation arithmetic and precision, not its assumptions or causal interpretation.

### What this cell does

- Plots a heatmap with the percentage chance that each draft slot finishes in each final points position from 1 through 12.
- Compares expected finish under pooled, null, and recent slot effects.
- Compares modeled first-place probability with the observed historical rate.
- Shows how top-six probability changes across weekly-persistence assumptions.

In [5]:
# CELL [5 plot-simulation-results]
heatmap = baseline_probs
heatmap_text = [[f"{probability:.1%}" for probability in row] for row in heatmap]
fig_rank = go.Figure(go.Heatmap(
    z=heatmap,
    text=heatmap_text,
    texttemplate="%{text}",
    textfont={"size": 10},
    x=list(range(1, 13)),
    y=list(range(1, 13)),
    colorscale="Blues",
    colorbar={"title": "Chance", "tickformat": ".0%"},
    hovertemplate="Draft slot %{y}<br>Final points finish %{x}<br>Chance %{z:.2%}<extra></extra>",
))
fig_rank.update_layout(title="Chance of each final points finish by draft slot", xaxis_title="Final regular-season points position", yaxis_title="Draft position", template="plotly_white", height=680)
fig_rank.update_xaxes(dtick=1, side="top")
fig_rank.update_yaxes(dtick=1, autorange="reversed")

comparison_names = ["Pooled effects, ICC 0.15", "No slot effect, ICC 0.15", "2022-2025 effects, ICC 0.15"]
expected_plot = summary[summary["scenario"].isin(comparison_names)]
fig_expected = px.line(expected_plot, x="draft_slot", y="expected_finish", color="scenario", markers=True, title="Expected final points finish across slot-effect scenarios")
fig_expected.update_layout(template="plotly_white", height=500, xaxis_title="Draft slot", yaxis_title="Expected finish, lower is better", legend_title="Scenario")
fig_expected.update_yaxes(autorange="reversed")

observed_first = panel.groupby("draft_slot")["top_regular_season_scorer"].mean().reindex(range(1, 13))
first_compare = pd.DataFrame({
    "draft_slot": list(range(1, 13)) * 2,
    "probability": np.concatenate([baseline_summary.sort_values("draft_slot")["first_place_probability"].to_numpy(), observed_first.to_numpy()]),
    "series": ["Model, pooled effects"] * 12 + ["Observed panel"] * 12,
})
fig_first = px.bar(first_compare, x="draft_slot", y="probability", color="series", barmode="group", title="First overall by points: model versus observed panel")
fig_first.update_layout(template="plotly_white", height=500, xaxis_title="Draft slot", yaxis_title="Probability", legend_title="Series")
fig_first.update_yaxes(tickformat=".1%")

persistence_names = ["Pooled effects, ICC 0.05", "Pooled effects, ICC 0.15", "Pooled effects, ICC 0.30"]
persistence_plot = summary[summary["scenario"].isin(persistence_names)]
fig_persistence = px.line(persistence_plot, x="draft_slot", y="top_6_probability", color="scenario", markers=True, title="Top-six points probability under weekly-persistence sensitivity")
fig_persistence.update_layout(template="plotly_white", height=500, xaxis_title="Draft slot", yaxis_title="Top-six probability", legend_title="Scenario")
fig_persistence.update_yaxes(tickformat=".1%")

figures = {
    "simulation_01_rank_distribution.html": fig_rank,
    "simulation_02_expected_finish.html": fig_expected,
    "simulation_03_first_overall.html": fig_first,
    "simulation_04_persistence_sensitivity.html": fig_persistence,
}
for filename, figure in figures.items():
    figure.write_html(ARTIFACTS / filename, include_plotlyjs="cdn")
    figure.show()

print({"plotly_charts_written": len(figures), "artifact_directory": str(ARTIFACTS)})

{'plotly_charts_written': 4, 'artifact_directory': 'C:\\Users\\josep\\Desktop\\random_stuff\\cowork_OS\\fantasy_draft_studies\\fantasy_draft_order_study\\artifacts'}


### Interpreting the output

- The baseline heatmap puts slot 4 at the best expected finish, while slot 1 shifts toward lower final ranks. Slot 4 reaches the top six 52.32% of the time versus 46.19% for slot 1.
- Using only 2022-2025 effects widens the expected-finish gap between slots 4 and 1 from 0.500 to 0.668 places.
- Changing weekly persistence from 0.05 to 0.30 moves slot 4's top-six probability by only 0.15 percentage points.
- The model-versus-observed chart is an in-sample fit diagnostic, not out-of-sample validation.

### What this cell does

- Writes the full rank distribution, slot summaries, assumptions, and evaluation gates.
- Displays the baseline slot summary with a polished-table library when available.
- Falls back to an interactive table or plain dataframe without making rendering packages execution requirements.

In [6]:
# CELL [6 save-artifacts-and-display]
summary_path = ARTIFACTS / "simulation_slot_summary.csv"
rank_path = ARTIFACTS / "simulation_rank_probabilities.csv"
assumptions_path = ARTIFACTS / "simulation_assumptions.json"
evaluation_path = ARTIFACTS / "simulation_evaluation.json"
summary.to_csv(summary_path, index=False)
rank_distribution.to_csv(rank_path, index=False)

assumptions = {
    "model_type": "17-week latent-score Monte Carlo",
    "simulation_is_causal": False,
    "teams_per_league": N_TEAMS,
    "regular_season_weeks": N_WEEKS,
    "simulated_league_seasons_per_scenario": N_SIMULATIONS,
    "random_seed": SEED,
    "weekly_shock_distribution": f"Student t with {T_DF} degrees of freedom, variance standardized",
    "weekly_scores_in_retained_data": False,
    "weekly_persistence_status": "assumed and sensitivity-tested, not estimated",
    "weekly_icc_scenarios": [LOW_ICC, BASELINE_ICC, HIGH_ICC],
    "effect_scenarios": ["pooled 2018-2025", "no slot effect", "2022-2025"],
    "residual_sd_from_season_points_zscore": residual_sd,
    "ranking_target": "final regular-season points only",
    "excluded": ["head-to-head wins", "schedules", "playoffs", "waivers", "trades"],
}
evaluation = {
    "historical_league_seasons": int(len(league_sizes)),
    "historical_team_seasons": int(len(panel)),
    "rank_accounting_gate": "pass",
    "null_symmetry_gate": "pass",
    "monte_carlo_precision_gate": "pass",
    "max_null_first_place_deviation_percentage_points": 100 * max_null_deviation,
    "max_first_place_mc_se_percentage_points": 100 * max_mc_se,
    "plotly_artifacts": list(figures),
}
assumptions_path.write_text(json.dumps(assumptions, indent=2), encoding="utf-8")
evaluation_path.write_text(json.dumps(evaluation, indent=2), encoding="utf-8")

baseline_display = baseline_summary[["draft_slot", "expected_finish", "median_finish", "top_6_probability", "first_place_probability", "first_place_mc_se"]].sort_values("draft_slot").copy()
baseline_display.columns = ["Draft slot", "Expected finish", "Median finish", "Top-six probability", "First-place probability", "First-place MC SE"]
try:
    from great_tables import GT
    display(GT(baseline_display).fmt_number(columns=["Expected finish"], decimals=3).fmt_percent(columns=["Top-six probability", "First-place probability", "First-place MC SE"], decimals=2))
except ImportError:
    try:
        from itables import init_notebook_mode, show
        init_notebook_mode(all_interactive=True)
        show(baseline_display)
    except ImportError:
        display(baseline_display)

slot_1 = baseline_summary.loc[baseline_summary["draft_slot"] == 1].iloc[0]
slot_4 = baseline_summary.loc[baseline_summary["draft_slot"] == 4].iloc[0]
print({
    "files_written": 4 + len(figures),
    "slot_1_expected_finish": round(float(slot_1["expected_finish"]), 3),
    "slot_4_expected_finish": round(float(slot_4["expected_finish"]), 3),
    "slot_1_first_place": round(float(slot_1["first_place_probability"]), 4),
    "slot_4_first_place": round(float(slot_4["first_place_probability"]), 4),
})

Draft slot,Expected finish,Median finish,Top-six probability,First-place probability,First-place MC SE
1,6.816,7.0,46.19%,7.17%,0.08%
2,6.501,7.0,49.93%,8.29%,0.09%
3,6.363,6.0,51.70%,9.10%,0.09%
4,6.316,6.0,52.32%,9.05%,0.09%
5,6.408,6.0,51.03%,8.66%,0.09%
6,6.395,6.0,51.31%,8.78%,0.09%
7,6.455,6.0,50.65%,8.32%,0.09%
8,6.423,6.0,50.85%,8.71%,0.09%
9,6.550,7.0,49.44%,8.20%,0.09%
10,6.678,7.0,47.84%,7.65%,0.08%


{'files_written': 8, 'slot_1_expected_finish': 6.816, 'slot_4_expected_finish': 6.316, 'slot_1_first_place': 0.0717, 'slot_4_first_place': 0.0905}


### Interpreting the output

- Eight artifacts were written: four data or metadata files and four Plotly HTML charts.
- Slot 4's expected finish is 6.316 versus 6.816 for slot 1. Their modeled first-place probabilities are 9.05% and 7.17%, respectively.
- The saved tables expose every scenario, slot, rank probability, and Monte Carlo standard error.
- These probabilities remain conditional on the model and the retained convenience sample.

## Conclusion

- Under the pooled model, slot 4 has the best expected regular-season points finish at 6.316, compared with 6.816 for slot 1. Slot 4's top-six probability is 52.32% versus 46.19% for slot 1.
- The notebook writes full rank probabilities, scenario summaries, assumptions, evaluation gates, and four Plotly charts to `artifacts/`.
- All structural, rank-accounting, null-symmetry, precision, and artifact gates passed.
- The result is model-based and does not settle causality, universal league representativeness, weekly persistence, head-to-head finish, or playoff outcomes.